# Cyclical EMA Strategy Research

Goal: 在周期性波动美股标的上，验证 EMA 趋势跟踪 + LightGBM gating 是否产生 Sharpe ≥ 1.0 且优于纯 EMA baseline ≥ +0.3 的策略。

Reference:
- Design: `docs/plans/2026-05-07-cyclical-ema-research-design.md`
- Impl plan: `docs/plans/2026-05-08-cyclical-ema-research-impl.md`

In [1]:
# === Cell 0: Imports + Constants ===
from __future__ import annotations
import os, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import optuna
import shap

from oxq.indicators.builtin import EMA, ATR, MFI, OBV
from oxq.indicators.hurst_exponent import HurstExponent
from oxq.indicators.annualized_volatility import AnnualizedVolatility
from oxq.indicators.rolling_volatility import RollingVolatility
from oxq.indicators.garch_volatility import GarchVolatility
from oxq.indicators.rolling_mdd import RollingMDD
from oxq.indicators.nday_return import NdayReturn

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

# --- Constants ---
def _find_repo_root() -> Path:
    """Walk up from cwd until we find pyproject.toml (the open-xquant repo marker)."""
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(
        "Cannot find repo root (no pyproject.toml in cwd or any parent). "
        "Run the notebook from somewhere inside the open-xquant repo tree."
    )

REPO_ROOT    = _find_repo_root()
PROJECT_ROOT = REPO_ROOT / "examples" / "research" / "cyclical_ema"
CACHE_DIR    = PROJECT_ROOT / "cache"
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# 数据刷新 flag —— 默认 False，迭代时使用缓存；改 True 强制重拉
REFRESH_DATA = False

# 时间锚点（来自 design doc 第 2.3 节）
ANCHOR_DATE   = pd.Timestamp("2026-05-07")            # universe filter as-of
TRAIN_CUTOFF  = pd.Timestamp("2021-05-07")            # 训练/测试切分
HISTORY_START = pd.Timestamp("2010-01-01")            # 数据起点（覆盖 ≥10y 的 universe）

# Universe 过滤阈值
MIN_LISTING_YEARS_STRICT = 10                          # A 组
MIN_LISTING_YEARS_RELAX  = 7                           # B 组
MIN_DOLLAR_VOL_STRICT    = 5_000_000                   # A 组
MIN_DOLLAR_VOL_RELAX     = 2_000_000                   # B 组
HURST_WINDOW             = 100
HURST_THRESHOLD_STRICT   = 0.40
HURST_THRESHOLD_RELAX    = 0.50
ANN_VOL_WINDOW           = 60
ANN_VOL_THRESHOLD_STRICT = 0.25
ANN_VOL_THRESHOLD_RELAX  = 0.20

# Sample 标签
LABEL_RETURN_THRESHOLD   = 0.05                        # gross_return ≥ 5% → 1
MIN_HISTORY_BARS         = 252                         # t_buy 之前最少交易日数

# 模型 + 回测
RANDOM_STATE             = 42
N_OPTUNA_TRIALS          = 50
N_CV_FOLDS               = 5
SCORE_THRESHOLD_X        = 0.55
TOP_N                    = 10
ROUND_TRIP_BPS           = 20                          # 0.20%

# 行业映射
SECTOR_TO_SPDR = {
    "Technology":              "XLK",
    "Financial Services":      "XLF",
    "Energy":                  "XLE",
    "Healthcare":              "XLV",
    "Industrials":             "XLI",
    "Consumer Defensive":      "XLP",
    "Consumer Cyclical":       "XLY",
    "Utilities":               "XLU",
    "Basic Materials":         "XLB",
    "Real Estate":             "XLRE",
    "Communication Services":  "XLC",
}
MARKET_TICKERS = ["^GSPC", "^VIX"] + list(set(SECTOR_TO_SPDR.values()))

print(f"Project root: {PROJECT_ROOT}")
print(f"REFRESH_DATA: {REFRESH_DATA}")
print(f"Anchor date: {ANCHOR_DATE.date()}, Train cutoff: {TRAIN_CUTOFF.date()}")


Project root: /Users/daodao/Documents/2-coding-space/git/github.com/open-xquant/examples/research/cyclical_ema
REFRESH_DATA: False
Anchor date: 2026-05-07, Train cutoff: 2021-05-07


In [2]:
# === Cell 1: Universe ticker list (Nasdaq Trader) ===
TICKER_CACHE = CACHE_DIR / "nasdaq_tickers.parquet"

def fetch_nasdaq_tickers() -> pd.DataFrame:
    """Fetch nasdaqlisted.txt + otherlisted.txt; return cleaned common-stock tickers."""
    base = "https://www.nasdaqtrader.com/dynamic/symdir"
    nq = pd.read_csv(f"{base}/nasdaqlisted.txt", sep="|")
    nq = nq[nq["Symbol"].notna() & (nq["Symbol"] != "File Creation Time")].copy()
    nq["Exchange"] = "NASDAQ"
    nq = nq[(nq["Test Issue"] == "N") & (nq["ETF"] == "N")]

    ot = pd.read_csv(f"{base}/otherlisted.txt", sep="|")
    ot = ot[ot["ACT Symbol"].notna() & (ot["ACT Symbol"] != "File Creation Time")].copy()
    ot = ot.rename(columns={"ACT Symbol": "Symbol"})
    ot = ot[(ot["Test Issue"] == "N") & (ot["ETF"] == "N")]

    df = pd.concat(
        [nq[["Symbol", "Security Name", "Exchange"]],
         ot[["Symbol", "Security Name", "Exchange"]]],
        ignore_index=True,
    )

    # 排除 ADR / 衍生品（启发式）
    bad_pattern = df["Symbol"].str.contains(r"[\.\$=]", regex=True, na=False)
    too_long = df["Symbol"].str.len() > 4
    adr_pattern = df["Security Name"].str.contains(
        r"American Depositary|ADR|ADS", case=False, regex=True, na=False)

    df = df[~bad_pattern & ~too_long & ~adr_pattern].reset_index(drop=True)
    return df

if REFRESH_DATA or not TICKER_CACHE.exists():
    tickers_df = fetch_nasdaq_tickers()
    tickers_df.to_parquet(TICKER_CACHE)
else:
    tickers_df = pd.read_parquet(TICKER_CACHE)

print(f"Total tickers after type filter: {len(tickers_df)}")
print(tickers_df.head(10))

assert 4000 < len(tickers_df) < 7000, f"Expected 4000-7000 tickers, got {len(tickers_df)}"
assert tickers_df["Symbol"].is_unique, "Symbols should be unique"


Total tickers after type filter: 5667
  Symbol                                      Security Name Exchange
0   AACB  Artius II Acquisition Inc. - Class A Ordinary ...   NASDAQ
1   AACI  Armada Acquisition Corp. III - Class A Ordinar...   NASDAQ
2   AACO  Abony Acquisition Corp. I - Class A Ordinary S...   NASDAQ
3    AAL       American Airlines Group, Inc. - Common Stock   NASDAQ
4   AAME       Atlantic American Corporation - Common Stock   NASDAQ
5   AAOI       Applied Optoelectronics, Inc. - Common Stock   NASDAQ
6   AAON                          AAON, Inc. - Common Stock   NASDAQ
7   AAPG  Ascentage Pharma Group International - America...   NASDAQ
8   AAPL                          Apple Inc. - Common Stock   NASDAQ
9   AARD         Aardvark Therapeutics, Inc. - Common Stock   NASDAQ
